In [1]:
import pandas as pd
import json

In [3]:
df = pd.read_csv(
    "01_District_wise_crimes_committed_IPC_2001_2012.csv"
)

df.head()

,STATE/UT,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,...,ARSON,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES
0,ANDHRA PRADESH,ADILABAD,2001,101,60,17,50,0,50,46,...,30,1131,16,149,34,175,0,181,1518,4154
1,ANDHRA PRADESH,ANANTAPUR,2001,151,125,1,23,0,23,53,...,69,1543,7,118,24,154,0,270,754,4125
2,ANDHRA PRADESH,CHITTOOR,2001,101,57,2,27,0,27,59,...,38,2088,14,112,83,186,0,404,1262,5818
3,ANDHRA PRADESH,CUDDAPAH,2001,80,53,1,20,0,20,25,...,23,795,17,126,38,57,0,233,1181,3140
4,ANDHRA PRADESH,EAST GODAVARI,2001,82,67,1,23,0,23,49,...,41,1244,12,109,58,247,0,431,2313,6507


In [4]:
df.columns = df.columns.str.strip()

df.columns.tolist()

['STATE/UT',
 'DISTRICT',
 'YEAR',
 'MURDER',
 'ATTEMPT TO MURDER',
 'CULPABLE HOMICIDE NOT AMOUNTING TO MURDER',
 'RAPE',
 'CUSTODIAL RAPE',
 'OTHER RAPE',
 'KIDNAPPING & ABDUCTION',
 'KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS',
 'KIDNAPPING AND ABDUCTION OF OTHERS',
 'DACOITY',
 'PREPARATION AND ASSEMBLY FOR DACOITY',
 'ROBBERY',
 'BURGLARY',
 'THEFT',
 'AUTO THEFT',
 'OTHER THEFT',
 'RIOTS',
 'CRIMINAL BREACH OF TRUST',
 'CHEATING',
 'COUNTERFIETING',
 'ARSON',
 'HURT/GREVIOUS HURT',
 'DOWRY DEATHS',
 'ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY',
 'INSULT TO MODESTY OF WOMEN',
 'CRUELTY BY HUSBAND OR HIS RELATIVES',
 'IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES',
 'CAUSING DEATH BY NEGLIGENCE',
 'OTHER IPC CRIMES',
 'TOTAL IPC CRIMES']

In [5]:
crime_columns = [
    col for col in df.columns
    if col not in ["STATE/UT", "DISTRICT", "YEAR"]
]

df["TOTAL_CRIME"] = df[crime_columns].sum(axis=1)

df.head()

,STATE/UT,DISTRICT,YEAR,MURDER,ATTEMPT TO MURDER,CULPABLE HOMICIDE NOT AMOUNTING TO MURDER,RAPE,CUSTODIAL RAPE,OTHER RAPE,KIDNAPPING & ABDUCTION,...,HURT/GREVIOUS HURT,DOWRY DEATHS,ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY,INSULT TO MODESTY OF WOMEN,CRUELTY BY HUSBAND OR HIS RELATIVES,IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES,CAUSING DEATH BY NEGLIGENCE,OTHER IPC CRIMES,TOTAL IPC CRIMES,TOTAL_CRIME
0,ANDHRA PRADESH,ADILABAD,2001,101,60,17,50,0,50,46,...,1131,16,149,34,175,0,181,1518,4154,8603
1,ANDHRA PRADESH,ANANTAPUR,2001,151,125,1,23,0,23,53,...,1543,7,118,24,154,0,270,754,4125,8692
2,ANDHRA PRADESH,CHITTOOR,2001,101,57,2,27,0,27,59,...,2088,14,112,83,186,0,404,1262,5818,12445
3,ANDHRA PRADESH,CUDDAPAH,2001,80,53,1,20,0,20,25,...,795,17,126,38,57,0,233,1181,3140,6498
4,ANDHRA PRADESH,EAST GODAVARI,2001,82,67,1,23,0,23,49,...,1244,12,109,58,247,0,431,2313,6507,14107


In [6]:
top_states = (
    df.groupby("STATE/UT")["TOTAL_CRIME"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_states

STATE/UT
MADHYA PRADESH    10278890
MAHARASHTRA       10273532
ANDHRA PRADESH     8719740
TAMIL NADU         8655534
UTTAR PRADESH      8213038
RAJASTHAN          7982662
KARNATAKA          6354578
GUJARAT            6008504
KERALA             5895218
BIHAR              5792604
Name: TOTAL_CRIME, dtype: int64

In [7]:
yearly_trend = (
    df.groupby("YEAR")["TOTAL_CRIME"]
    .sum()
    .sort_index()
)

yearly_trend

YEAR
2001     7659962
2002     7692690
2003     7426632
2004     7957250
2005     7919012
2006     8148558
2007     8625374
2008     9110494
2009     9244284
2010     9681172
2011    10121640
2012    10368596
Name: TOTAL_CRIME, dtype: int64

In [8]:
zone_distribution = {
    "Critical": 0,
    "High": 0,
    "Moderate": 0,
    "Controlled": 0
}

for value in top_states.values:

    if value > 1000000:
        zone_distribution["Critical"] += 1

    elif value > 500000:
        zone_distribution["High"] += 1

    elif value > 200000:
        zone_distribution["Moderate"] += 1

    else:
        zone_distribution["Controlled"] += 1

zone_distribution

{'Critical': 10, 'High': 0, 'Moderate': 0, 'Controlled': 0}

In [9]:
growth = yearly_trend.diff()

fastest_growth_year = int(growth.idxmax())

In [10]:
regional_summary = {
    "most_dangerous_state": str(top_states.idxmax()),
    "highest_crime_records": int(top_states.max()),
    "fastest_growth_year": fastest_growth_year,
    "total_states_analyzed": int(df["STATE/UT"].nunique()),
    "insight":
        "High-density metropolitan and industrial regions "
        "showed significantly elevated crime accumulation "
        "throughout the processed NCRB dataset."
}

regional_summary

{'most_dangerous_state': 'MADHYA PRADESH',
 'highest_crime_records': 10278890,
 'fastest_growth_year': 2004,
 'total_states_analyzed': 35,
 'insight': 'High-density metropolitan and industrial regions showed significantly elevated crime accumulation throughout the processed NCRB dataset.'}

In [11]:
processed = {
    "top_states": top_states.to_dict(),
    "yearly_trend": yearly_trend.to_dict(),
    "zone_distribution": zone_distribution,
    "regional_summary": regional_summary
}

processed

{'top_states': {'MADHYA PRADESH': 10278890,
  'MAHARASHTRA': 10273532,
  'ANDHRA PRADESH': 8719740,
  'TAMIL NADU': 8655534,
  'UTTAR PRADESH': 8213038,
  'RAJASTHAN': 7982662,
  'KARNATAKA': 6354578,
  'GUJARAT': 6008504,
  'KERALA': 5895218,
  'BIHAR': 5792604},
 'yearly_trend': {2001: 7659962,
  2002: 7692690,
  2003: 7426632,
  2004: 7957250,
  2005: 7919012,
  2006: 8148558,
  2007: 8625374,
  2008: 9110494,
  2009: 9244284,
  2010: 9681172,
  2011: 10121640,
  2012: 10368596},
 'zone_distribution': {'Critical': 10,
  'High': 0,
  'Moderate': 0,
  'Controlled': 0},
 'regional_summary': {'most_dangerous_state': 'MADHYA PRADESH',
  'highest_crime_records': 10278890,
  'fastest_growth_year': 2004,
  'total_states_analyzed': 35,
  'insight': 'High-density metropolitan and industrial regions showed significantly elevated crime accumulation throughout the processed NCRB dataset.'}}

In [12]:
with open(
    "../frontend/src/data/states_processed.json",
    "w"
) as f:
    json.dump(processed, f, indent=4)

print("states_processed.json exported successfully")

states_processed.json exported successfully


In [17]:
import os

print(os.listdir())

['01_District_wise_crimes_committed_IPC_2001_2012.csv', 'crime_records_analysis.ipynb', 'crime_records_processed.json', 'node_modules', 'package-lock.json', 'package.json', 'server.js', 'states_analysis.ipynb']


In [18]:
import pandas as pd
import json

# =========================================
# LOAD DATASET
# =========================================

df = pd.read_csv("01_District_wise_crimes_committed_IPC_2001_2012.csv")

# =========================================
# CLEAN COLUMN NAMES
# =========================================

df.columns = [col.strip().upper() for col in df.columns]

# =========================================
# REQUIRED COLUMNS
# =========================================

STATE_COL = "STATE/UT"
DISTRICT_COL = "DISTRICT"
YEAR_COL = "YEAR"

# =========================================
# CRIME COLUMNS
# =========================================

crime_columns = []

for col in df.columns:

    if col not in [STATE_COL, DISTRICT_COL, YEAR_COL]:

        if df[col].dtype != "object":

            crime_columns.append(col)

# =========================================
# TOTAL CRIME
# =========================================

df["TOTAL_CRIME"] = df[crime_columns].sum(axis=1)

# =========================================
# TOP STATES
# =========================================

state_totals = (
    df.groupby(STATE_COL)["TOTAL_CRIME"]
    .sum()
    .sort_values(ascending=False)
)

# =========================================
# BUILD FINAL DATA
# =========================================

states_data = {}

for state in state_totals.index[:15]:

    state_df = df[df[STATE_COL] == state]

    district_data = {}

    districts = (
        state_df.groupby(DISTRICT_COL)["TOTAL_CRIME"]
        .sum()
        .sort_values(ascending=False)
    )

    for district in districts.index[:10]:

        district_df = state_df[
            state_df[DISTRICT_COL] == district
        ]

        total_crime = int(
            district_df["TOTAL_CRIME"].sum()
        )

        yearly = (
            district_df.groupby(YEAR_COL)["TOTAL_CRIME"]
            .sum()
            .sort_index()
        )

        crime_totals = {}

        for col in crime_columns:

            crime_totals[col] = district_df[col].sum()

        top_crime_type = max(
            crime_totals,
            key=crime_totals.get
        )

        district_data[district] = {

            "total_crime": total_crime,

            "top_crime": top_crime_type,

            "yearly_trend": yearly.to_dict()
        }

    states_data[state] = district_data

# =========================================
# FINAL JSON
# =========================================

final_data = {
    "states": states_data
}

# =========================================
# EXPORT JSON
# =========================================

with open(
    "../frontend/src/data/states_processed.json",
    "w"
) as f:

    json.dump(final_data, f, indent=4)

print("states_processed.json exported successfully")

states_processed.json exported successfully
